In [2]:
import numpy as np
import networkx as nx
import pickle
import os
import json

from utils import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

setup_environment()
set_paper_style()

In [8]:
import json
import pandas as pd

# --- 設定 ---
INPUT_JSON_PATH = "knowledge_graph_with_wiki.json"
OUTPUT_NODES_CSV = "graphrag_project/input/nodes.csv"
OUTPUT_EDGES_CSV = "graphrag_project/input/edges.csv"

# --- 知識グラフの読み込み ---
with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    graph_data = json.load(f)


def build_node_text(node: dict) -> str:
    """RAG 用の自然言語説明だけを抽出する。"""
    summary = node.get("summary_text")
    if summary:
        return summary.strip()
    return node.get("label", "")


# --- ノードデータの作成 ---
node_records = []
for node in graph_data["nodes"]:
    if node.get("type") == "市区町村":
        node_records.append(
            {
                "id": node["id"],
                "label": node.get("label"),
                "type": node.get("type"),
                "text": build_node_text(node),
                "properties_json": json.dumps(node.get("properties", {}), ensure_ascii=False),
            }
        )

df_nodes = pd.DataFrame(node_records)
df_nodes.to_csv(OUTPUT_NODES_CSV, index=False, encoding="utf-8")
print(f"✅ {len(df_nodes)}件のノードデータを '{OUTPUT_NODES_CSV}' に保存しました。")

# --- エッジデータの作成 ---
edge_records = []
for edge in graph_data["links"]:
    if edge.get("label") == "IS_IN":
        edge_records.append(
            {
                "source": edge["source"],
                "target": edge["target"],
                "description": f"{edge['source']}は{edge['target']}に属しています。",
            }
        )

df_edges = pd.DataFrame(edge_records)
df_edges.to_csv(OUTPUT_EDGES_CSV, index=False, encoding="utf-8")
print(f"✅ {len(df_edges)}件のエッジデータを '{OUTPUT_EDGES_CSV}' に保存しました。")


✅ 1741件のノードデータを 'graphrag_project/input/nodes.csv' に保存しました。
✅ 1741件のエッジデータを 'graphrag_project/input/edges.csv' に保存しました。


In [9]:
import subprocess
import os

# https://microsoft.github.io/graphrag/config/models/
# 約9時間ぐらいかかってて草(o4-mini)
# ==============================================================================
# 設定
# ==============================================================================
# GraphRagプロジェクトのルートフォルダ
PROJECT_ROOT = "./graphrag_project"

# ユーザーからの質問
USER_QUESTION = "温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？"
# ==============================================================================


def run_command(command):
    """
    ターミナルコマンドを実行し、出力をリアルタイムで表示する関数
    """
    print(f"▶️ 実行中: {' '.join(command)}\n")
    # `subprocess.run` を使用してコマンドを実行
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        # .envファイルから環境変数を読み込むために、PROJECT_ROOTをカレントディレクトリとして指定
        cwd=os.path.abspath(PROJECT_ROOT),
    )

    # 標準出力と標準エラー出力を表示
    if result.stdout:
        print("--- 出力 ---")
        print(result.stdout)
    if result.stderr:
        print("--- エラー出力 ---")
        print(result.stderr)

    if result.returncode == 0:
        print("\n✅ コマンドは正常に完了しました。")
    else:
        print(
            f"\n❌ コマンドの実行中にエラーが発生しました (終了コード: {result.returncode})。"
        )

    return result


# --- 1. インデックスの作成 ---
print("🔄 インデックス作成を開始します...（データ量に応じて時間がかかります）")
# 正しいコマンド: `graphrag index`
index_command = ["graphrag", "index", "--root", "."]
run_command(index_command)

🔄 インデックス作成を開始します...（データ量に応じて時間がかかります）
▶️ 実行中: graphrag index --root .

--- 出力 ---
Starting pipeline with workflows: load_input_documents, create_base_text_units, create_final_documents, extract_graph, finalize_graph, extract_covariates, create_communities, create_final_text_units, create_community_reports, generate_text_embeddings
Starting workflow: load_input_documents

Workflow complete: load_input_documents
Starting workflow: create_base_text_units
  1 / 1 ............................................................................................
  1 / 1 ............................................................................................
  1 / 1 ............................................................................................
  1 / 1 ............................................................................................
  1 / 1 ............................................................................................
  1 / 1 ...............................

CompletedProcess(args=['graphrag', 'index', '--root', '.'], returncode=1, stdout='Starting pipeline with workflows: load_input_documents, create_base_text_units, create_final_documents, extract_graph, finalize_graph, extract_covariates, create_communities, create_final_text_units, create_community_reports, generate_text_embeddings\nStarting workflow: load_input_documents\n\nWorkflow complete: load_input_documents\nStarting workflow: create_base_text_units\n  1 / 1 ............................................................................................\n  1 / 1 ............................................................................................\n  1 / 1 ............................................................................................\n  1 / 1 ............................................................................................\n  1 / 1 ............................................................................................\n  1 / 1 .....................

In [ ]:
import subprocess
import os
# ==============================================================================
# 設定
# ==============================================================================
PROJECT_ROOT = "./graphrag_project"
# USER_QUESTION = "現在、福岡市に住んでいて福岡市の良いところを維持しながら２拠点生活をしたと考えている。また、飛行機を利用したいと思っているため、九州以外での２拠点都市を考えている。特徴としては、温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？"

# ==============================================================================
print(f"\n【質問】: {USER_QUESTION}")
print("🔄 GraphRagで検索を実行しています...")

# '--query' オプションを質問文の前に追加
query_command = [
    "graphrag",
    "query",
    "--root",
    ".",
    "--method",
    "global",
    # "local",
    "--query",
    USER_QUESTION,
]
query_result = run_command(query_command)


【質問】: 現在、福岡市に住んでいて福岡市の良いところを維持しながら２拠点生活をしたと考えている。また、飛行機を利用したいと思っているため、九州以外での２拠点都市を考えている。特徴としては、温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？
🔄 GraphRagで検索を実行しています...
▶️ 実行中: graphrag query --root . --method global --query 現在、福岡市に住んでいて福岡市の良いところを維持しながら２拠点生活をしたと考えている。また、飛行機を利用したいと思っているため、九州以外での２拠点都市を考えている。特徴としては、温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？

--- 出力 ---
以下は、福岡市に居住しつつ「九州以外で温泉地・自然が豊かで静かな生活を送るための二拠点生活」を検討する際に、有力とされる候補を、データセットに基づいて整理したものです。九州以外の拠点として温泉資源が明確に記され、自然と静かな暮らしが想定される地域を中心に抜粋しています。あなたの条件を満たす候補を複数提示し、それぞれの根拠データを併記します。なお、二拠点生活の実現性には、飛行機を含む移動手段の利便性が関係しますが、データには地域ごとに異なる移動手段の前提が示されている点を踏まえて検討してください。

1) Nozawa Onsen Village（長野県、野沢温泉）  
- 要点: 九州外の拠点として最も分かりやすい温泉地のひとつ。温泉資源が豊かで自然環境に恵まれ、静かな暮らしが期待できます。  
- データ参照例: Nozawa Onsen Village may serve as one base outside Kyushu for a quiet, nature-rich lifestyle, as it is recognized for its hot springs and is located in Nagano Prefecture’s northeast region. [Data: Reports (319)]

2) Hakone region（神奈川県）  
- 要点: 箱根は温泉資源が非常に充実しており、芦ノ湖・自然景観・観光資源が揃っている一方、静かな暮らしを

In [4]:
import subprocess
import os

# https://microsoft.github.io/graphrag/config/models/
# 約9時間ぐらいかかってて草(o4-mini)
# ==============================================================================
# 設定
# ==============================================================================
# GraphRagプロジェクトのルートフォルダ
PROJECT_ROOT = "./graphrag_project"
# ==============================================================================


def run_command(command):
    """
    ターミナルコマンドを実行し、出力をリアルタイムで表示する関数
    """
    print(f"▶️ 実行中: {' '.join(command)}\n")
    # `subprocess.run` を使用してコマンドを実行
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        # .envファイルから環境変数を読み込むために、PROJECT_ROOTをカレントディレクトリとして指定
        cwd=os.path.abspath(PROJECT_ROOT),
    )

    # 標準出力と標準エラー出力を表示
    if result.stdout:
        print("--- 出力 ---")
        print(result.stdout)
    if result.stderr:
        print("--- エラー出力 ---")
        print(result.stderr)

    if result.returncode == 0:
        print("\n✅ コマンドは正常に完了しました。")
    else:
        print(
            f"\n❌ コマンドの実行中にエラーが発生しました (終了コード: {result.returncode})。"
        )

    return result

import subprocess
import os
# ==============================================================================
# 設定
# ==============================================================================
PROJECT_ROOT = "./graphrag_project"
# USER_QUESTION = "現在、福岡市に住んでいて福岡市の良いところを維持しながら２拠点生活をしたと考えている。また、飛行機を利用したいと思っているため、九州以外での２拠点都市を考えている。特徴としては、温泉地として有名で、自然が豊かで静かに暮らせそうな街はどこですか？"
USER_QUESTION = "名古屋と教育・医療・福祉のレベルで似た市町村は？"
# ==============================================================================
print(f"\n【質問】: {USER_QUESTION}")
print("🔄 GraphRagで検索を実行しています...")

# '--query' オプションを質問文の前に追加
query_command = [
    "graphrag",
    "query",
    "--root",
    ".",
    "--method",
    "global",
    # "local",
    "--query",
    USER_QUESTION,
]
query_result = run_command(query_command)


【質問】: 名古屋と教育・医療・福祉のレベルで似た市町村は？
🔄 GraphRagで検索を実行しています...
▶️ 実行中: graphrag query --root . --method global --query 名古屋と教育・医療・福祉のレベルで似た市町村は？

--- 出力 ---
I am sorry but I am unable to answer this question given the provided data.

--- エラー出力 ---
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x3a92b13d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x3afb1d790>


✅ コマンドは正常に完了しました。


In [12]:
# --- 3. 結果の表示 ---
# 検索結果は`query_result.stdout`に格納されている
print("\n\n===================================")
print("      ✨ GraphRagによる回答 ✨")
print("===================================")
# 回答はJSON形式で返ってくることが多いので、そのまま表示
print(query_result.stdout)



      ✨ GraphRagによる回答 ✨

